In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/REAL DATA/dataset/tts_data"
CSV_PATH  = f"{DATA_ROOT}/metadata.csv"


In [ ]:
!pip -q install sentencepiece protobuf hf_transfer
!pip -q install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install snac torchcodec "datasets>=3.4.1,<4.0.0" torchaudio


In [ ]:
#Build Dataset

import os, glob
import pandas as pd
from datasets import Dataset, Audio

CLEAR_ROOT = "/content/drive/MyDrive/REAL DATA/dataset/clear"
TRANSCRIPTS_DIR = "/content/drive/MyDrive/REAL DATA/dataset/transcripts"

def read_text_for_utt(speaker_dir, utt_id):

    cand1 = os.path.join(speaker_dir, f"{utt_id}.txt")
    if os.path.exists(cand1):
        return open(cand1, "r", encoding="utf-8").read().strip()

    if TRANSCRIPTS_DIR is not None:
        cand2 = os.path.join(TRANSCRIPTS_DIR, f"{utt_id}.txt")
        if os.path.exists(cand2):
            return open(cand2, "r", encoding="utf-8").read().strip()

    return None

rows = []
speaker_dirs = sorted([d for d in glob.glob(os.path.join(CLEAR_ROOT, "speaker_*")) if os.path.isdir(d)])

for sp_dir in speaker_dirs:
    spk = os.path.basename(sp_dir)
    wavs = sorted(glob.glob(os.path.join(sp_dir, "*.wav")))
    for wav_path in wavs:
        utt_id = os.path.splitext(os.path.basename(wav_path))[0]
        text = read_text_for_utt(sp_dir, utt_id)
        if not text:
            continue
        rows.append({"source": spk, "text": text, "audio": wav_path})

df = pd.DataFrame(rows)
print("Total rows:", len(df), " | speakers:", df["source"].nunique())
print(df.head())

dataset = Dataset.from_pandas(df, preserve_index=False)
dataset = dataset.cast_column("audio", Audio(sampling_rate=None))


In [ ]:
import os
import pandas as pd
from datasets import Dataset, Audio

df = pd.read_csv(CSV_PATH)  # expects columns: audio_path,text
df["audio"] = df["audio_path"].apply(lambda p: os.path.join(DATA_ROOT, p))
df = df[["text", "audio"]]

dataset = Dataset.from_pandas(df, preserve_index=False)
dataset = dataset.cast_column("audio", Audio(sampling_rate=None))  # keep original; we resample later
print(dataset[0].keys())
print(dataset[0]["audio"]["sampling_rate"], dataset[0]["text"])

In [ ]:
import torchaudio.transforms as T
import torch
from snac import SNAC

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda")

def tokenise_audio(audio_array, sr):
    waveform = torch.from_numpy(audio_array).unsqueeze(0).to(dtype=torch.float32)  # [1, T]
    if sr != 24000:
        waveform = T.Resample(orig_freq=sr, new_freq=24000)(waveform)
    waveform = waveform.unsqueeze(0).to("cuda")  # [1, 1, T]

    with torch.inference_mode():
        codes = snac_model.encode(waveform)

    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(codes[0][0][i].item() + 128266)
        all_codes.append(codes[1][0][2*i].item() + 128266 + 4096)
        all_codes.append(codes[2][0][4*i].item() + 128266 + (2*4096))
        all_codes.append(codes[2][0][(4*i)+1].item() + 128266 + (3*4096))
        all_codes.append(codes[1][0][(2*i)+1].item() + 128266 + (4*4096))
        all_codes.append(codes[2][0][(4*i)+2].item() + 128266 + (5*4096))
        all_codes.append(codes[2][0][(4*i)+3].item() + 128266 + (6*4096))
    return all_codes

def add_codes(example):
    try:
        a = example["audio"]
        example["codes_list"] = tokenise_audio(a["array"], a["sampling_rate"])
    except Exception as e:
        print("Skipping row:", e)
        example["codes_list"] = None
    return example

dataset_tok = dataset.map(add_codes, remove_columns=["audio"])
dataset_tok = dataset_tok.filter(lambda x: x["codes_list"] is not None and len(x["codes_list"]) > 0)
print("Kept rows:", len(dataset_tok))

def remove_duplicate_frames(example):
    vals = example["codes_list"]
    if len(vals) % 7 != 0:
        # drop malformed rows
        example["codes_list"] = None
        return example

    result = vals[:7]
    for i in range(7, len(vals), 7):
        if vals[i] != result[-7]:
            result.extend(vals[i:i+7])
    example["codes_list"] = result
    return example


In [ ]:
tokeniser_length = 128256
end_of_text = 128009
start_of_human = tokeniser_length + 3
end_of_human   = tokeniser_length + 4
start_of_ai    = tokeniser_length + 5
end_of_ai      = tokeniser_length + 6
start_of_speech = tokeniser_length + 1
end_of_speech   = tokeniser_length + 2

dataset_tok = dataset_tok.map(remove_duplicate_frames)
dataset_tok = dataset_tok.filter(lambda x: x["codes_list"] is not None and len(x["codes_list"]) > 0)

def create_input_ids(example):
    text_prompt = example["text"]
    text_ids = tokenizer.encode(text_prompt, add_special_tokens=True)
    text_ids.append(end_of_text)

    input_ids = (
        [start_of_human]
        + text_ids
        + [end_of_human]
        + [start_of_ai]
        + [start_of_speech]
        + example["codes_list"]
        + [end_of_speech]
        + [end_of_ai]
    )
    example["input_ids"] = input_ids
    example["labels"] = input_ids
    example["attention_mask"] = [1] * len(input_ids)
    return example

dataset_final = dataset_tok.map(create_input_ids, remove_columns=["text", "codes_list"])
keep = {"input_ids", "labels", "attention_mask"}
drop_cols = [c for c in dataset_final.column_names if c not in keep]
dataset_final = dataset_final.remove_columns(drop_cols)

print(dataset_final[0].keys())
print("Rows:", len(dataset_final))
print("Seq len example:", len(dataset_final[0]["input_ids"]))


In [ ]:
from transformers import TrainingArguments, Trainer

trainer = Trainer(
    model=model,
    train_dataset=dataset_final,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        max_steps=1200,
        learning_rate=2e-5,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=f"{DATA_ROOT}/orpheus_outputs_clear15",
        report_to="none",
        fp16=True,
    ),
)

trainer.train()


In [ ]:
from IPython.display import display, Audio
import torch

FastLanguageModel.for_inference(model)
snac_model.to("cpu")

prompts = [
    "masigla"
]

start_token = torch.tensor([[128259]], dtype=torch.int64)              # SOH
end_tokens  = torch.tensor([[128009, 128260]], dtype=torch.int64)      # EOT, EOH

all_modified = []
for p in prompts:
    ids = tokenizer(p, return_tensors="pt").input_ids
    all_modified.append(torch.cat([start_token, ids, end_tokens], dim=1))

max_len = max(x.shape[1] for x in all_modified)
padded, masks = [], []
for x in all_modified:
    pad = max_len - x.shape[1]
    padded.append(torch.cat([torch.full((1, pad), 128263, dtype=torch.int64), x], dim=1))
    masks.append(torch.cat([torch.zeros((1, pad), dtype=torch.int64), torch.ones((1, x.shape[1]), dtype=torch.int64)], dim=1))

input_ids = torch.cat(padded, dim=0).to("cuda")
attn_mask = torch.cat(masks, dim=0).to("cuda")

gen = model.generate(
    input_ids=input_ids,
    attention_mask=attn_mask,
    max_new_tokens=1200,
    do_sample=True,
    temperature=0.6,
    top_p=0.95,
    repetition_penalty=1.1,
    num_return_sequences=1,
    eos_token_id=128258,
    use_cache=True,
)

# Crop to after token 128257, remove token 128258
token_to_find = 128257
token_to_remove = 128258
idx = (gen == token_to_find).nonzero(as_tuple=True)
cropped = gen[:, idx[1][-1].item()+1:] if len(idx[1]) > 0 else gen
rows = [row[row != token_to_remove] for row in cropped]

code_lists = []
for row in rows:
    new_len = (row.size(0) // 7) * 7
    trimmed = row[:new_len].tolist()
    trimmed = [t - 128266 for t in trimmed]
    code_lists.append(trimmed)

def redistribute_codes(code_list):
    layer_1, layer_2, layer_3 = [], [], []
    for i in range((len(code_list)+1)//7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i+1]-4096)
        layer_3.append(code_list[7*i+2]-(2*4096))
        layer_3.append(code_list[7*i+3]-(3*4096))
        layer_2.append(code_list[7*i+4]-(4*4096))
        layer_3.append(code_list[7*i+5]-(5*4096))
        layer_3.append(code_list[7*i+6]-(6*4096))
    codes = [torch.tensor(layer_1).unsqueeze(0),
             torch.tensor(layer_2).unsqueeze(0),
             torch.tensor(layer_3).unsqueeze(0)]
    return snac_model.decode(codes)

for p, cl in zip(prompts, code_lists):
    audio_hat = redistribute_codes(cl)
    print(p)
    display(Audio(audio_hat.detach().squeeze().cpu().numpy(), rate=24000))


In [ ]:
SAVE_DIR = f"{DATA_ROOT}/orpheus_lora_tagalog"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved LoRA to:", SAVE_DIR)

In [ ]:
################
LOAD MODE
################

from unsloth import FastLanguageModel
import torch

LORA_PATH = "/content/drive/MyDrive/REAL DATA/dataset/tts_data/orpheus_lora_tagalog"  # change if needed

# Load base model again
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/orpheus-3b-0.1-ft",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)

# Attach your LoRA adapters
model.load_adapter(LORA_PATH)

# Switch to inference mode (important!)
FastLanguageModel.for_inference(model)

model = model.to("cuda")
print("Model + LoRA loaded successfully.")

In [ ]:
from snac import SNAC

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz")
snac_model = snac_model.to("cpu")  # decoding on CPU is fine

In [ ]:
from IPython.display import Audio, display
import torch

def generate_speech(text, chosen_voice=None):

    if chosen_voice:
        text = f"{chosen_voice}: {text}"

    start_token = torch.tensor([[128259]], dtype=torch.int64)
    end_tokens  = torch.tensor([[128009, 128260]], dtype=torch.int64)

    input_ids = tokenizer(text, return_tensors="pt").input_ids
    input_ids = torch.cat([start_token, input_ids, end_tokens], dim=1).to("cuda")

    attention_mask = torch.ones_like(input_ids)

    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1200,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        repetition_penalty=1.1,
        eos_token_id=128258,
        use_cache=True
    )

    # Find speech token start
    token_to_find = 128257
    idx = (generated_ids == token_to_find).nonzero(as_tuple=True)

    if len(idx[1]) > 0:
        generated_ids = generated_ids[:, idx[1][-1].item()+1:]

    generated_ids = generated_ids[generated_ids != 128258]

    # Convert tokens back to SNAC codes
    trimmed = generated_ids[: (generated_ids.shape[0] // 7) * 7]
    trimmed = [t.item() - 128266 for t in trimmed]

    layer_1, layer_2, layer_3 = [], [], []
    for i in range((len(trimmed)+1)//7):
        layer_1.append(trimmed[7*i])
        layer_2.append(trimmed[7*i+1]-4096)
        layer_3.append(trimmed[7*i+2]-(2*4096))
        layer_3.append(trimmed[7*i+3]-(3*4096))
        layer_2.append(trimmed[7*i+4]-(4*4096))
        layer_3.append(trimmed[7*i+5]-(5*4096))
        layer_3.append(trimmed[7*i+6]-(6*4096))

    codes = [
        torch.tensor(layer_1).unsqueeze(0),
        torch.tensor(layer_2).unsqueeze(0),
        torch.tensor(layer_3).unsqueeze(0),
    ]

    audio_hat = snac_model.decode(codes)

    display(Audio(audio_hat.detach().squeeze().numpy(), rate=24000))


In [ ]:
generate_speech("masakit ang tiyan ko")